# Battery Credit / Thermal Score Model (Ultimate Optimization & Data Augmentation)

This notebook implements the **Ultimate Model Pipeline** for the NASA Battery Dataset. We address the small sample size (34 batteries) using **Data Augmentation** and optimize **Random Forest**, **XGBoost**, and **LightGBM** using **Grid Search**.

## 🚀 Optimization Strategies
1. **Data Augmentation**: We expanded the dataset from 34 to 340 samples by adding controlled Gaussian noise to the extracted features. This helps the models generalize beyond specific IDs.
2. **Hyperparameter Tuning**: We performed a full `GridSearchCV` on all models to find the best splitting constraints and learning rates.
3. **Accuracy Metric**: We defined "Accuracy" as the percentage of predictions falling within ±2.5 points of the ground truth score.

In [62]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

BASE_DIR = r'c:\Users\Hp\OneDrive\Desktop\credit-score\cleaned_dataset'
DATA_DIR = os.path.join(BASE_DIR, 'data')
METADATA_PATH = os.path.join(BASE_DIR, 'metadata.csv')

def calculate_accuracy(y_true, y_pred, tolerance=2.5):
    errors = np.abs(y_true - y_pred)
    return np.mean(errors <= tolerance) * 100

print("Ready for ultimate optimization.")

Ready for ultimate optimization.


## 1. Feature Extraction & Data Augmentation

In [ ]:
def extract_features(group):
    filenames = [str(f) for f in group['filename'].dropna().tolist()]
    all_temps = []
    total_duration_over_40 = 0
    fast_charge_count = 0
    deep_discharge_count = 0
    
    discharge_cycles = group[group['type'] == 'discharge']
    capacities = discharge_cycles['Capacity'].dropna().tolist()
    capacity_fade_rate = (capacities[0] - capacities[-1]) / len(capacities) if len(capacities) > 1 else 0

    for fname in filenames:
        fpath = os.path.join(DATA_DIR, fname)
        if not os.path.exists(fpath): continue
        try:
            df = pd.read_csv(fpath)
            if 'Temperature_measured' in df.columns:
                temps = pd.to_numeric(df['Temperature_measured'], errors='coerce').dropna().tolist()
                all_temps.extend(temps)
                total_duration_over_40 += len([t for t in temps if t > 40])
            if 'Voltage_measured' in df.columns:
                if (pd.to_numeric(df['Voltage_measured'], errors='coerce') < 2.7).any(): deep_discharge_count += 1
            if 'Current_measured' in df.columns:
                if (pd.to_numeric(df['Current_measured'], errors='coerce').abs() > 1.5).any(): fast_charge_count += 1
        except: continue

    return pd.Series({
        'avg_operating_temp': np.mean(all_temps) if all_temps else 0,
        'max_temp': np.max(all_temps) if all_temps else 0,
        'overtemp_count': len([t for t in all_temps if t > 40]) / max(len(filenames), 1),
        'overtemp_duration_log': np.log1p(total_duration_over_40),
        'temp_variance': np.var(all_temps) if all_temps else 0,
        'fast_charge_ratio': fast_charge_count / max(len(filenames), 1),
        'deep_discharge_ratio': deep_discharge_count / max(len(filenames), 1),
        'capacity_fade_rate': capacity_fade_rate
    })

def augment_data(df, multiplier=10, noise_level=0.01):
    augmented_list = [df]
    for _ in range(multiplier - 1):
        noisy_copy = df.copy()
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        numeric_cols = [c for c in numeric_cols if c != 'Battery_Credit_Score']
        for col in numeric_cols:
            noise = np.random.normal(0, noise_level * df[col].std(), size=len(df))
            noisy_copy[col] = noisy_copy[col] + noise
        augmented_list.append(noisy_copy)
    return pd.concat(augmented_list, ignore_index=True)

print("Starting processing...")
metadata = pd.read_csv(METADATA_PATH)
metadata['Capacity'] = pd.to_numeric(metadata['Capacity'], errors='coerce')

feature_list = []
for b_id in metadata['battery_id'].unique():
    group = metadata[metadata['battery_id'] == b_id]
    features = extract_features(group)
    features['battery_id'] = b_id
    feature_list.append(features)

df_base = pd.DataFrame(feature_list)

def calculate_score(row):
    score = 100
    score -= 0.4 * row['overtemp_count']
    score -= 0.005 * np.expm1(row['overtemp_duration_log'])
    score -= 15 * row['fast_charge_ratio']
    score -= 20 * row['deep_discharge_ratio']
    score -= 10 * (row['capacity_fade_rate'] * 100)
    return np.clip(score, 0, 100)

df_base['Battery_Credit_Score'] = df_base.apply(calculate_score, axis=1)
df_aug = augment_data(df_base, multiplier=10)


Starting processing...
Database expanded from 34 to 340 samples.


## 2. Final Optimized Training

In [66]:
X = df_aug.drop(['battery_id', 'Battery_Credit_Score'], axis=1)
y = df_aug['Battery_Credit_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "RandomForest": GridSearchCV(RandomForestRegressor(random_state=42), {'n_estimators': [100, 200], 'max_depth': [5, 10]}, cv=5),
    "XGBoost": GridSearchCV(XGBRegressor(random_state=42), {'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}, cv=5),
    "LightGBM": GridSearchCV(LGBMRegressor(verbose=-1, random_state=42), {'learning_rate': [0.05, 0.1], 'min_child_samples': [1, 5]}, cv=5)
}

print("Training optimized models...")
for name, grid in models.items():
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    acc = calculate_accuracy(y_test, y_pred, tolerance=2.5)
    print(f"{name:12} -> RMSE: {rmse:.4f}, Accuracy: {acc:.2f}%")
    
    # Save the ultimate version
    if name == "LightGBM":
        joblib.dump(best_model, 'battery_credit_model_lightGBM.pkl')
        print("Ultimate Model Saved.")

Training optimized models...
RandomForest -> RMSE: 5.5519, Accuracy: 98.53%
XGBoost      -> RMSE: 2.1118, Accuracy: 95.59%
LightGBM     -> RMSE: 0.4405, Accuracy: 98.53%
Ultimate Model Saved.
